# Frame 1 (Day -2) Morphology vs. Eventual Fate Mixture - vem 1uM, 48hrs Pre-Treatment

This analysis only ever covers frame 1 (2 days before the drug is
added - `config.FRAMES = [1]`) and asks: does a cell's *shape* at that
early timepoint relate to what its lineage will eventually become by
day 0?

Frame-1 cells don't have a single subclass label - `cloneClassification`
only classifies cells present at day 0 (frame 3), and by then a frame-1
lineage can have already split into several day-0 descendants with
*different* fates. So each frame-1 cell gets a **mixture**: the fraction
of its day-0 descendants that end up dying / persisting / slow
proliferator / fast proliferator, from `lineage_frame3_to_frame1.mat`
and `clone_kmeans_id.mat` (see `lineage_fate.py`).

This notebook is purely visual (no statistical test) - a weighted KDE
per fate category, where each frame-1 cell contributes to every
category's contour in proportion to its own percentage in that category.
Plots both embeddings `main.py` computes (UMAP and PHATE) every time
this notebook runs - no manual switch to flip, both are always kept up
to date, each under its own subfolder.

Requires `main.py` to have already been run (for `selected_metadata.csv`
and `{cell,nuc}_umap.npz`/`{cell,nuc}_phate.npz` under
`output/cajal_output/`), and `config.RAW_DRUG_DIR` / `LINEAGE_MAT_PATH` /
`CLONE_KMEANS_MAT_PATH` to point at the real raw data.


In [ ]:
import os
import sys

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/2_vem_48hr_pretreatment/notebooks/), so the analysis dir - and
# config.py - is one level up, and the repo root - where shared/ lives -
# is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

import matplotlib.pyplot as plt

from shared.load_data import load_full_dataset
from shared.density_plots import plot_embedding_with_weighted_kde
import lineage_fate


## Method

For each embedding method (UMAP, PHATE), then each modality (cell, nucleus):

1. **Load the already-computed frame-1 embedding** (`load_full_dataset`) -
   `main.py` only ever runs CAJAL/UMAP/PHATE on frame-1 cells now, so this
   is already exactly the right data, no subsetting needed.
2. **Resolve each frame-1 lineage's fate mixture**:
   - `lineage_fate.build_raw_to_global_lin_id` resolves
     `lineage_frame3_to_frame1.mat`'s track-local
     `(file_idx, old_clone_id_frame1)` IDs into the same global `lin_id`
     already in `{drug}_metadata.csv`, by directly reusing
     process_imaging_data's own `load_track`/`make_master_ids`.
   - `lineage_fate.compute_frame1_fate_percentages` then gives each
     frame-1 lineage's `pct_dying`/`pct_persisting`/`pct_slow_prolif`/`pct_fast_prolif`.
     This only depends on the raw `.mat` files, not on which embedding
     method is being plotted, so it's computed once and reused for both.
   - A frame-1 cell whose lineage dies out before reaching day 0 has no
     entry here (no day-0 descendants to classify) - it gets 0% in every
     category, i.e. it contributes to none of the four contours, rather
     than being dropped from the plot entirely.
3. **Plot**: one weighted KDE contour per fate category over the frame-1
   embedding, background scatter in plain grey
   (`plot_embedding_with_weighted_kde`, levels 0.5/0.7/0.9).


In [ ]:
SUBCLASS_COLORS = {
    'dying':       'tab:red',
    'persisting':  'tab:purple',
    'slow_prolif': 'tab:orange',
    'fast_prolif': 'tab:green',
}
WEIGHT_COLS = {
    'dying':       'pct_dying',
    'persisting':  'pct_persisting',
    'slow_prolif': 'pct_slow_prolif',
    'fast_prolif': 'pct_fast_prolif',
}
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '01_frame1_fate_mixture')

# main.py computes both of these for every modality - plot both every
# time this notebook runs, rather than picking just one
EMBEDDING_METHODS = ['umap', 'phate']


def process_modality(modality, fate_pct):
    '''
    For each embedding method, load this modality's already-computed
    frame-1 embedding, attach each frame-1 cell's fate-mixture
    percentages, and plot the weighted KDE.

    Inputs:
        modality: 'cell' or 'nuc'
        fate_pct: from lineage_fate.compute_frame1_fate_percentages
    Outputs:
        dfs: {embedding_method: frame-1 cells' metadata + dim_0/dim_1 +
              pct_* columns for that embedding method} (in-memory only)
    '''
    dfs = {}
    for embedding_method in EMBEDDING_METHODS:
        df = load_full_dataset(config.CAJAL_OUTPUT_DIR, modality, embedding_method)
        print(f"[{modality}/{embedding_method}] {len(df)} frame-1 cells")

        df = df.merge(fate_pct, on='lin_id', how='left')
        pct_cols = list(WEIGHT_COLS.values())
        n_dropped = df[pct_cols[0]].isna().sum()
        if n_dropped > 0:
            print(f"  Dropping {n_dropped}/{len(df)} frame-1 cell(s) with no day-0 descendants in "
                  f"lineage_frame3_to_frame1.mat (lineage died before day 0)")
        df = df.dropna(subset=pct_cols).reset_index(drop=True)

        plots_dir = os.path.join(PLOTS_DIR, embedding_method)
        os.makedirs(os.path.join(plots_dir, 'pdfs'), exist_ok=True)
        os.makedirs(os.path.join(plots_dir, 'svgs'), exist_ok=True)

        fig, ax = plt.subplots(figsize=(6, 6))
        plot_embedding_with_weighted_kde(
            df, WEIGHT_COLS, SUBCLASS_COLORS, ax,
            f'{modality} - frame 1 (day -2) fate mixture', embedding_method.upper(),
        )
        fig.savefig(os.path.join(plots_dir, 'pdfs', f'{modality}_frame1_fate_mixture.pdf'), bbox_inches='tight')
        fig.savefig(os.path.join(plots_dir, 'svgs', f'{modality}_frame1_fate_mixture.svg'), bbox_inches='tight')
        plt.close(fig)
        print(f"  Saved to {plots_dir}")

        dfs[embedding_method] = df

    return dfs


raw_to_global = lineage_fate.build_raw_to_global_lin_id(config.RAW_DRUG_DIR)
fate_pct = lineage_fate.compute_frame1_fate_percentages(
    config.LINEAGE_MAT_PATH, config.CLONE_KMEANS_MAT_PATH, raw_to_global,
)
print(f"{len(fate_pct)} frame-1 lineages have at least one day-0 descendant")


## Run: cell morphology

In [ ]:
cell_dfs = process_modality('cell', fate_pct)
cell_dfs['umap'].head()


## Run: nuclear morphology

In [ ]:
nuc_dfs = process_modality('nuc', fate_pct)
nuc_dfs['umap'].head()
